# Urdu Question Generation - training run

From-scratch BiLSTM encoder-decoder with Bahdanau attention.

**Before running:** Settings panel -> Accelerator = **GPU** (P100 or T4 x2), Internet = **On**.

This notebook is a thin driver: every real step lives in the repo's `src/` modules so the
code is identical locally and here. Order: install -> clone -> data prep -> tokenizer ->
debug gate -> full train -> evaluate -> zip outputs.

In [ ]:
# 1. Environment. torch + datasets are preinstalled on Kaggle images.
!pip -q install sentencepiece sacrebleu rouge-score
import torch, subprocess
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2. Get the code. Set REPO_URL to the public repo.
REPO_URL = 'https://github.com/Hanzala-12/urdu-question-generation.git'  # <-- confirm
import os, sys, pathlib
WORK = pathlib.Path('/kaggle/working')
REPO = WORK / 'repo'
if not REPO.exists():
    !git clone --depth 1 $REPO_URL {REPO}
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('cwd:', os.getcwd())
print(sorted(p.name for p in (REPO / 'src').glob('*.py')))

In [ ]:
# 3. Task 1 - data preparation (needs Internet: On). ~2-4 min.
!python -m src.data_prep
!wc -l data/*.tsv

In [ ]:
# 4. Task 2 - SentencePiece tokenizer (vocab 8k). ~1-2 min.
!python -m src.spm_train

In [ ]:
# 5. Debug gate - 10k pairs, 1 epoch. Loss MUST fall or there is a bug.
!python -m src.train --debug

In [ ]:
# 6. Task 3 - full training. ~1-2 h on P100 (batch 64, 15 epochs).
!python -m src.train --epochs 15 --batch-size 64

In [ ]:
# 7. Task 4 - evaluation on UQA-valid and Wiki-UQA (greedy + beam).
!python -m src.evaluate --split both --beam-max 3000
import json; print(json.dumps(json.load(open('results/metrics.json')), indent=2, ensure_ascii=False))

In [ ]:
# 8. Package everything the local repo needs back.
import shutil, pathlib
OUT = pathlib.Path('/kaggle/working/outputs')
OUT.mkdir(exist_ok=True)
for sub in ['artifacts', 'results']:
    dst = OUT / sub
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(sub, dst)
shutil.make_archive('/kaggle/working/outputs', 'zip', OUT)
print('wrote /kaggle/working/outputs.zip')
!du -sh /kaggle/working/outputs.zip && ls -R /kaggle/working/outputs